<a href="https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected, ready to query.")

Connected, ready to query.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before building any rule, I want to check two signals first.

Signal 1: does CTR actually drop as position gets worse?
This is the same idea behind FlyRank's real CTR-fix flag, so I'm checking
it directly on my own data before trusting it.

Signal 2: does having more search impressions actually mean more real
click opportunity?
This connects to the quick-win idea, a page is only a "quick win" if
fixing it actually moves real numbers, and that depends on how much
traffic (impressions) it's already getting.

My rule: if a page ranks reasonably well (not
stuck deep in the results) but its CTR is below what other pages at that
same position normally get, and it has enough impressions to matter, flag
it for a CTR review.

Reason code this rule can output: ctr_below_expected_for_position
Action labels: review_ctr_fix or no_action

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: CTR vs position, does CTR really drop as position gets worse?
import pandas as pd
signal1 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'top_3'
            WHEN gsc_avg_position <= 10 THEN 'page_1'
            WHEN gsc_avg_position <= 20 THEN 'page_2'
            ELSE 'deep'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY avg_ctr DESC
""").df()
print("Signal 1 - CTR by position bucket:")
print(signal1)
print("\nVerdict: CONFIRMED - CTR clearly drops as position gets worse.")

# Signal 2: does more impression volume mean more real click opportunity?
base = con.sql(f"""
    SELECT gsc_impressions, gsc_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

base["impression_quartile"] = pd.qcut(base["gsc_impressions"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])

signal2_grouped = base.groupby("impression_quartile").agg(
    n=("gsc_impressions", "count"),
    avg_impressions=("gsc_impressions", "mean"),
    avg_clicks=("gsc_clicks", "mean")
).reset_index()

print("Signal 2 - impression quartile vs avg clicks:")
print(signal2_grouped)
print("\nVerdict: CONFIRMED - higher impression volume clearly means more real click opportunity.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 - CTR by position bucket:
  position_bucket        n   avg_ctr
0           top_3   727362  0.004756
1          page_1  1456122  0.003473
2          page_2   519223  0.002770
3            deep   908354  0.001289

Verdict: CONFIRMED - CTR clearly drops as position gets worse.
Signal 2 - impression quartile vs avg clicks:
  impression_quartile       n  avg_impressions  avg_clicks
0              Q1_low  973112         2.106352    0.007227
1                  Q2  868861         9.329519    0.022046
2                  Q3  873000        33.927440    0.095276
3             Q4_high  896088       268.816414    0.795087

Verdict: CONFIRMED - higher impression volume clearly means more real click opportunity.


/tmp/ipykernel_736/2601761255.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2_grouped = base.groupby("impression_quartile").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Now I'm turning the confirmed signals into one simple score.

The rule: take a page's CTR, compare it to what's "expected" for its
position bucket (using the bucket averages from Signal 1), and flag pages
that are underperforming their position AND have enough impressions to be
worth fixing.

Score = expected_ctr_for_position - actual_ctr (a bigger gap = worse
underperformance = higher priority)

I'm also requiring impressions >= 50, so I'm not flagging pages with so
little traffic that the CTR number is basically noise.

Reason code: ctr_below_expected_for_position
Action label: review_ctr_fix if the gap is positive and impressions are
high enough, otherwise no_action.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

expected_ctr = signal1.set_index("position_bucket")["avg_ctr"].to_dict()

full = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 50
""").df()

def bucket(pos):
    if pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "page_2"
    else: return "deep"

full["position_bucket"] = full["gsc_avg_position"].apply(bucket)
full["actual_ctr"] = full["gsc_clicks"] / full["gsc_impressions"]
full["expected_ctr"] = full["position_bucket"].map(expected_ctr)
full["ctr_gap"] = full["expected_ctr"] - full["actual_ctr"]

full["reason_code"] = "ctr_below_expected_for_position"
full["action"] = full["ctr_gap"].apply(lambda g: "review_ctr_fix" if g > 0 else "no_action")

ranked = full.sort_values(["ctr_gap", "gsc_impressions"], ascending=[False, False]).reset_index(drop=True)
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} ranked rows to work/outputs/baseline_action_score.csv")
print(ranked[["content_hash_id", "gsc_avg_position", "actual_ctr", "expected_ctr", "ctr_gap", "action"]].head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 1037442 ranked rows to work/outputs/baseline_action_score.csv
            content_hash_id  gsc_avg_position  actual_ctr  expected_ctr  \
0  content_fec55986a1868d62          0.181500         0.0      0.004756   
1  content_44f34c0a90047651          0.132532         0.0      0.004756   
2  content_fec55986a1868d62          0.083407         0.0      0.004756   
3  content_9c057b66c30a3abb          0.000311         0.0      0.004756   
4  content_9c057b66c30a3abb          0.002245         0.0      0.004756   
5  content_9c057b66c30a3abb          0.317996         0.0      0.004756   
6  content_757b1fa67827358d          2.261644         0.0      0.004756   
7  content_fec55986a1868d62          0.013056         0.0      0.004756   
8  content_8e1334d6356668e3          0.665895         0.0      0.004756   
9  content_8e1334d6356668e3          0.397117         0.0      0.004756   

    ctr_gap          action  
0  0.004756  review_ctr_fix  
1  0.004756  review_ctr_fix  
2  0.004756  rev

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing my top 20 flagged pages. For each one: what action it got, why
it's there, and what would prove this pick wrong.

General pattern across most of these: they're in strong positions (mostly
top_3, some higher up) but have literally 0 clicks despite getting real
impressions. The action is review_ctr_fix for all of them, since the CTR
gap is at its maximum (expected_ctr - 0).

What would make any of these wrong: if the "0 clicks" number is actually a
data artifact (e.g., tracking not properly attributing clicks for that
page), rather than a real user behavior problem. I can't fully rule that
out from this data alone, so I'm treating these as a review shortlist,
not confirmed problems.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)[["content_hash_id", "gsc_avg_position", "gsc_impressions", "actual_ctr", "expected_ctr", "ctr_gap", "action"]]
print(top20.to_string(index=True))

# Check which content_hash_ids repeat in the top 20
repeat_counts = top20["content_hash_id"].value_counts()

print("\nReview notes:")
for i, row in top20.iterrows():
    cid = row['content_hash_id']
    times_repeated = repeat_counts[cid]
    if times_repeated > 1:
        note = (f"This page shows up {times_repeated} times in my top 20, always with 0 clicks — "
                f"that repetition makes me more suspicious this is a tracking issue rather than "
                f"a real CTR problem I could fix with content changes.")
    else:
        note = (f"Position {row['gsc_avg_position']:.2f} with {row['gsc_impressions']:.0f} impressions "
                f"and 0 clicks is unusual for such a strong ranking. Would be wrong if this page's "
                f"actual search intent doesn't match what people expect from that query, or if "
                f"click tracking simply isn't firing correctly on this page.")
    print(f"{i}: {row['action']} - {note}")

             content_hash_id  gsc_avg_position  gsc_impressions  actual_ctr  expected_ctr   ctr_gap          action
0   content_fec55986a1868d62          0.181500            33383         0.0      0.004756  0.004756  review_ctr_fix
1   content_44f34c0a90047651          0.132532            32958         0.0      0.004756  0.004756  review_ctr_fix
2   content_fec55986a1868d62          0.083407            31472         0.0      0.004756  0.004756  review_ctr_fix
3   content_9c057b66c30a3abb          0.000311            28973         0.0      0.004756  0.004756  review_ctr_fix
4   content_9c057b66c30a3abb          0.002245            28947         0.0      0.004756  0.004756  review_ctr_fix
5   content_9c057b66c30a3abb          0.317996            24233         0.0      0.004756  0.004756  review_ctr_fix
6   content_757b1fa67827358d          2.261644            19301         0.0      0.004756  0.004756  review_ctr_fix
7   content_fec55986a1868d62          0.013056            17770         

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: a few of my top 20 look almost too clean to be a normal CTR
problem, pages at position ~0.0-0.3 (essentially #1) with tens of
thousands of impressions and exactly 0 clicks. In real life, even a
mediocre #1 result gets some clicks. This pattern repeating across
multiple days for the same content_hash_id (e.g., content_fec55986a1868d62
shows 0 clicks on 4 different days) makes me suspect this might be a
tracking or data issue for those specific pages, not a genuine CTR
problem I could "fix" with content changes.

Leakage check: my rule only uses gsc_impressions, gsc_clicks, and
gsc_avg_position, all of these are things known at the actual moment of
measurement, not future outcomes. I did not use any product-decision flags
(nothing like "already flagged for review" or "already fixed"), and I did
not use any future time window, everything comes from the same
month=2026-03 slice I defined my contract around in the last assignment.
So I'm confident there's no leakage in this rule.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

repeat_check = ranked.head(20)["content_hash_id"].value_counts()
print("How many times each top-20 content_hash_id repeats in the top 20:")
print(repeat_check)

print("\nColumns used in the rule:", ["gsc_impressions", "gsc_clicks", "gsc_avg_position"])
print("None of these are future-window or product-decision fields — all pulled from the same month=2026-03 slice.")

How many times each top-20 content_hash_id repeats in the top 20:
content_hash_id
content_fec55986a1868d62    5
content_8e1334d6356668e3    4
content_9c057b66c30a3abb    3
content_bf078007df823490    3
content_44f34c0a90047651    1
content_757b1fa67827358d    1
content_69379902126ff53f    1
content_dc91779c3d085398    1
content_fa4cf3aa5ce67bb8    1
Name: count, dtype: int64

Columns used in the rule: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
None of these are future-window or product-decision fields — all pulled from the same month=2026-03 slice.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.